# FinovaTech v10.3.8 en Google Colab

Sistema de finanzas personales para estudiantes (Flask + MariaDB).

**Antes de empezar:** en tu equipo ejecuta `colab\empaquetar_colab.ps1`
para generar `FinovaTech_v10.3.8_Colab.zip` en el Escritorio. Luego sube ese
ZIP cuando este notebook lo pida.

> La sesión de Colab es temporal: al cerrarla se borran la base de datos y los
> cambios. Sirve para demostrar el proyecto, no para producción.

Ejecuta las celdas en orden con `Ctrl+F9` o con el botón de play de cada una.

## Paso 1. Subir y descomprimir el proyecto

In [ ]:
import glob
import os
import zipfile

from google.colab import files

os.makedirs('/content/proyecto', exist_ok=True)
print('Selecciona FinovaTech_v10.3.8_Colab.zip')
subidos = files.upload()
nombre_zip = next(iter(subidos))

with zipfile.ZipFile(nombre_zip) as z:
    z.extractall('/content/proyecto')

candidatos = [p for p in glob.glob('/content/proyecto/*') if os.path.isdir(p)]
assert candidatos, 'El ZIP no contiene una carpeta de proyecto'
PROYECTO = candidatos[0]
os.chdir(PROYECTO)
print('Proyecto listo en:', PROYECTO)
print('Archivos:', ', '.join(sorted(os.listdir('.')[:8])), '...')

## Paso 2. Instalar y arrancar MariaDB

Colab no trae XAMPP: se instala MariaDB (compatible con el esquema del
proyecto) dentro de la máquina temporal de Colab.

In [ ]:
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mariadb-server
!service mariadb start || service mysql start
!sleep 5
!mariadb --version
!mariadb-admin ping

### Crear base, usuario dedicado, cargar esquema y escribir `.env`

La clave es solo para esta sesión de Colab.

In [ ]:
import secrets
import subprocess

CLAVE = 'finova_colab_2026'

def sql(consulta):
    subprocess.run(['mariadb', '-uroot', '-e', consulta], check=True)

sql("CREATE DATABASE IF NOT EXISTS finovatech CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci")
sql("CREATE USER IF NOT EXISTS 'finovatech_app'@'localhost' IDENTIFIED BY '%s'" % CLAVE)
sql("CREATE USER IF NOT EXISTS 'finovatech_app'@'127.0.0.1' IDENTIFIED BY '%s'" % CLAVE)
sql("GRANT ALL PRIVILEGES ON finovatech.* TO 'finovatech_app'@'localhost'")
sql("GRANT ALL PRIVILEGES ON finovatech.* TO 'finovatech_app'@'127.0.0.1'")
sql("FLUSH PRIVILEGES")

subprocess.run('mariadb -uroot finovatech < database/schema.sql', shell=True, check=True)

env = [
    'DB_HOST=127.0.0.1',
    'DB_PORT=3306',
    'DB_USER=finovatech_app',
    'DB_PASSWORD=' + CLAVE,
    'DB_NAME=finovatech',
    'SECRET_KEY=' + secrets.token_hex(32),
    'PORT=5001',
    'FLASK_DEBUG=0',
]
open('.env', 'w', encoding='utf-8').write('\n'.join(env) + '\n')

sql("SELECT COUNT(*) AS tablas FROM information_schema.tables WHERE table_schema='finovatech'")
print('Base finovatech creada y .env escrito')

## Paso 3. Instalar dependencias, cargar datos demo y probar

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.seed_demo

In [ ]:
!python -m pytest -q

## Paso 4. Arrancar el servidor y abrir la aplicación

El servidor queda en segundo plano. La vista incrustada usa el proxy de
puertos de Colab.

In [ ]:
import os
import socket
import subprocess
import sys
import time

entorno = dict(os.environ)
entorno['HOST'] = '0.0.0.0'
entorno['PORT'] = '5001'
log = open('/tmp/finovatech.log', 'w')
servidor = subprocess.Popen([sys.executable, 'run.py'], stdout=log, stderr=subprocess.STDOUT, env=entorno)

for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', 5001), timeout=1):
            break
    except OSError:
        time.sleep(1)

print(open('/tmp/finovatech.log', encoding='utf-8', errors='replace').read()[-1500:])

from google.colab import output
output.serve_kernel_port_as_iframe(5001, height=820, path='/')
print('Si la vista no carga, abre la ventana con:')
print('  output.serve_kernel_port_as_window(5001)')

In [ ]:
# Detener el servidor cuando termines
try:
    servidor.terminate()
    print('Servidor detenido')
except NameError:
    print('El servidor no estaba arrancado')

## Cuentas demo

- Administrador/estudiante: `adminr123@gmail.com` / `123456789`
- Padre vinculado: `padredemo@gmail.com` / `123456789`

Recorrido sugerido: portada → login → panel → lista → reportes (PDF y PNG)
→ notificaciones → agendamiento → supervisión del padre → bitácora.

## Si algo falla

- **MariaDB no arranca:** ejecuta `!mysqld_safe --datadir=/var/lib/mysql &`
  y espera 5 segundos; luego repite el paso 2.
- **`Can't connect to MySQL`:** revisa que el paso 2 haya terminado sin error
  y que `.env` exista (`DB_HOST=127.0.0.1`, `DB_PORT=3306`).
- **La vista incrustada queda en blanco:** usa
  `output.serve_kernel_port_as_window(5001)` en una celda nueva.
- **Se reinició la sesión:** vuelve a ejecutar todo desde el paso 1; los
  datos de Colab no se conservan.
- **El ZIP no aparece:** genéralo con `colab\empaquetar_colab.ps1` en Windows.

> SMTP no está configurado en Colab: la recuperación de contraseña escribe el
> enlace en el log del servidor (`/tmp/finovatech.log`), no envía correos.